# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = ''

In [3]:
# DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/AF/obs'
DATA_PATH = '../../../updated-data/obs'
NULL_DATA_PATH = '../../../updated-data/null'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [5]:
fs = [
    f for f in os.listdir(NULL_DATA_PATH)
    if ('xling-' not in f)
    and (not f.startswith('.'))
    and (f.endswith('.parquet'))
    and (CORPUS_NAME in f)
]
len(fs)

3822

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'avar'

In [8]:
time_col = 'mtau'
# time_col = 'n'

In [9]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = table.read().to_pandas()
    df['self'] = df['x_speaker'] == df['y_speaker']
    df[COEF_VAR] = df[COEF_VAR] #/ df['n']

    # df['mtau'] = (df['min_y_turn_id'] + df['mtau']) - df['x_turn_id']
    # df[COEF_VAR] = np.sqrt(df[COEF_VAR])
    groups = df.groupby(by=['x_speaker', 'y_speaker'])
    # groups = df.groupby(by=['x_turn_id', 'y_speaker'])
    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        # dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        av = dfi[COEF_VAR].values
        mtau = dfi[time_col].values
        sel = (av > 0) & (mtau > 1)

        try:
            if sel.sum() > 1:

                b, a = np.polyfit(
                    np.log(mtau[sel]),
                    np.log(av[sel]),
                    1
                )

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'cid': f.replace('.parquet', ''),
                        'speaker': labels[0],
                        'self': self_other,
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

        except Exception:
            error_tracker += 1

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/3822 [00:00<?, ?it/s]

In [10]:
df_regression.isna().mean()

corpus     0.0
cid        0.0
speaker    0.0
self       0.0
a          0.0
b          0.0
k          0.0
dtype: float64

In [11]:
np.isinf(df_regression['b']).mean()

np.float64(0.0)

In [ ]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [12]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'null-all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [10]:
split_by = ['corpus', 'self']

In [11]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [12]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [13]:
df0.head(n=50)

,corpus,self,b,std,k,se,t,p
0,CABNC,False,0.265524,0.332775,5876,0.004341,61.163755,0.000000e+00
1,CABNC,True,0.293004,0.290355,2117,0.006311,46.430490,0.000000e+00
2,CANDOR,False,0.604234,0.164438,3310,0.002858,211.406162,0.000000e+00
3,CANDOR,True,0.605765,0.170659,3310,0.002966,204.215089,0.000000e+00
4,CORAAL,False,0.664452,0.282149,634,0.011206,59.296599,6.282995e-261
5,CORAAL,True,0.692373,0.232103,532,0.010063,68.804244,5.520196e-267
6,DISPEL,False,0.415171,0.227732,32,0.040258,10.312847,7.639005e-12
7,DISPEL,True,0.395001,0.229809,32,0.040625,9.723122,3.119775e-11
8,Frederiksen,False,0.000000,0.000000,2,0.000000,NaN,NaN
9,Frederiksen,True,0.000000,NaN,1,NaN,NaN,NaN


In [ ]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [13]:
split_by = ['self']

In [14]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [15]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [16]:
df0.head(n=20)

,self,b,std,k,se,t,p
0,False,0.154471,0.571919,12576,0.005100,30.288914,7.101465e-195
1,True,0.163967,0.211415,6709,0.002581,63.525745,0.000000e+00


In [17]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'null-results.csv'),
    index=False,
    encoding='utf-8'
)